In [ ]:
!pip install tensorflow opencv-python matplotlib numpy pandas

In [ ]:
import tensorflow as tf
import os

class DatasetLoader:
    def __init__(self, dataset_path, target_size=(128, 128), batch_size=32, has_subfolders=True, shuffle=True):
        """
        Initializes the dataset loader.

        Args:
            dataset_path (str): Path to the dataset folder.
            target_size (tuple): Target size for resizing images.
            batch_size (int): Number of images per batch.
            has_subfolders (bool): Whether dataset has subfolders for classes.
            shuffle (bool): Whether to shuffle the dataset.
        """
        self.dataset_path = dataset_path
        self.target_size = target_size
        self.batch_size = batch_size
        self.has_subfolders = has_subfolders
        self.shuffle = shuffle

        self.dataset, self.class_mapping = self._load_dataset()

    def _process_image(self, file_path, label):
        """
        Reads and processes an image.

        Args:
            file_path (str): Path to the image file.
            label (int): Numeric label for the image.

        Returns:
            (tf.Tensor, tf.Tensor): Processed image and its label.
        """
        image = tf.io.read_file(file_path)
        image = tf.image.decode_jpeg(image, channels=3)
        image = tf.image.resize(image, self.target_size) / 255.0  # Normalize
        return image, label

    def _load_dataset(self):
        """
        Loads dataset images and assigns numeric labels automatically.

        Returns:
            dataset (tf.data.Dataset): TensorFlow dataset.
            class_mapping (dict): Mapping from class names to numeric labels.
        """
        image_paths = []
        labels = []

        if self.has_subfolders:
            # Load dataset with subfolders as class labels
            class_names = sorted(os.listdir(self.dataset_path))
            class_mapping = {name: idx for idx, name in enumerate(class_names)}

            for class_name in class_names:
                class_folder = os.path.join(self.dataset_path, class_name)
                if os.path.isdir(class_folder):
                    for img_file in os.listdir(class_folder):
                        image_paths.append(os.path.join(class_folder, img_file))
                        labels.append(class_mapping[class_name])
        else:
            # Load dataset with no subfolders (single class)
            image_paths = [os.path.join(self.dataset_path, f) for f in os.listdir(self.dataset_path)]
            labels = [0] * len(image_paths)  # Assign a single label

            class_mapping = {"all": 0}

        # Convert lists to TensorFlow tensors
        image_paths_tensor = tf.constant(image_paths)
        labels_tensor = tf.constant(labels, dtype=tf.int32)

        # Create dataset
        dataset = tf.data.Dataset.from_tensor_slices((image_paths_tensor, labels_tensor))
        dataset = dataset.map(self._process_image, num_parallel_calls=tf.data.experimental.AUTOTUNE)

        if self.shuffle:
            dataset = dataset.shuffle(1000)

        dataset = dataset.batch(self.batch_size).prefetch(tf.data.experimental.AUTOTUNE)

        return dataset, class_mapping

    def get_dataset(self):
        """
        Returns the dataset.

        Returns:
            (tf.data.Dataset, dict): The TensorFlow dataset and class mapping.
        """
        return self.dataset.unbatch(), self.class_mapping

In [ ]:
import random


def create_balanced_pairs(dataset, class_mapping, samples_per_class=100):
    # Create reverse mapping (index to class name)
    reverse_mapping = {v: k for k, v in class_mapping.items()}

    # Convert dataset to list since we'll sample randomly
    dataset_list = list(dataset.as_numpy_iterator())

    # Group images by class index
    class_images = {class_idx: [] for class_idx in class_mapping.values()}
    for img, lbl in dataset_list:
        class_idx = lbl.item()  # Convert numpy to Python int
        class_images[class_idx].append(img)

    pairs = []
    labels = []
    class_indices = list(class_mapping.values())

    # Create positive pairs (same class)
    for class_idx in class_indices:
        images = class_images[class_idx]
        if len(images) < 2:
            continue

        for _ in range(samples_per_class):
            img1, img2 = random.sample(images, 2)
            pairs.append((img1, img2))
            labels.append(1)  # 1 = similar

    # Create negative pairs (different classes)
    for _ in range(samples_per_class * len(class_indices)):
        class1, class2 = random.sample(class_indices, 2)
        img1 = random.choice(class_images[class1])
        img2 = random.choice(class_images[class2])
        pairs.append((img1, img2))
        labels.append(0)  # 0 = dissimilar

    # Create dataset without batching
    def gen():
        for (img1, img2), label in zip(pairs, labels):
            yield (img1, img2), label

    return tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            (tf.TensorSpec(shape=(128, 128, 3), dtype=tf.float32),
             tf.TensorSpec(shape=(128, 128, 3), dtype=tf.float32)),
            tf.TensorSpec(shape=(), dtype=tf.int32)
        )
    ).shuffle(10000)

In [ ]:

from tensorflow.keras.applications import ResNet50, InceptionV3, MobileNetV2 # type: ignore
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Input # type: ignore
from tensorflow.keras.models import Model # type: ignore

class FeatureExtractor:
    def __init__(self, model_name="cnn", input_shape=(128, 128, 3), freeze=True):
        """
        Initializes the feature extractor.

        Args:
            model_name (str): The type of feature extractor ('cnn', 'resnet', 'inception', 'mobilenet').
            input_shape (tuple): The shape of input images.
        """
        self.model_name = model_name.lower()
        self.input_shape = input_shape
        self.freeze = freeze
        self.model = self._build_model()

    def _build_model(self):
        """Builds the selected feature extractor model."""
        if self.model_name == "resnet":
            return self._resnet()
        elif self.model_name == "inception":
            return self._inception()
        elif self.model_name == "mobilenet":
            return self._mobilenet()
        else:
            return self._cnn()  # Default CNN model

    def _cnn(self):
        """Simple custom CNN feature extractor."""
        inputs = Input(shape=self.input_shape)
        x = Conv2D(32, (3, 3), activation="relu", padding="same")(inputs)
        x = MaxPooling2D((2, 2))(x)
        x = Conv2D(64, (3, 3), activation="relu", padding="same")(x)
        x = MaxPooling2D((2, 2))(x)
        x = Flatten()(x)
        x = Dense(128, activation="relu")(x)
        model = Model(inputs, x, name="Custom_CNN")
        return model

    def _resnet(self):
        """ResNet50 as feature extractor (pretrained on ImageNet)."""
        base_model = ResNet50(weights="imagenet", include_top=False, input_shape=self.input_shape)
        if self.freeze:
            base_model.trainable = False
        x = Flatten()(base_model.output)
        model = Model(base_model.input, x, name="ResNet50_FeatureExtractor")
        return model

    def _inception(self):
        """InceptionV3 as feature extractor (pretrained on ImageNet)."""
        base_model = InceptionV3(weights="imagenet", include_top=False, input_shape=self.input_shape)
        x = Flatten()(base_model.output)
        model = Model(base_model.input, x, name="InceptionV3_FeatureExtractor")
        return model

    def _mobilenet(self):
        """MobileNetV2 as feature extractor (pretrained on ImageNet)."""
        base_model = MobileNetV2(weights="imagenet", include_top=False, input_shape=self.input_shape)
        x = Flatten()(base_model.output)
        model = Model(base_model.input, x, name="MobileNetV2_FeatureExtractor")
        return model

    def get_model(self):
        """Returns the built feature extractor model."""
        return self.model


In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential # type: ignore


class DistanceMetrics:
    @staticmethod
    def euclidean(embedding1, embedding2):
        """Computes Euclidean distance between two embeddings."""
        return tf.sqrt(tf.reduce_sum(tf.square(embedding1 - embedding2), axis=-1, keepdims=True))

    @staticmethod
    def manhattan(embedding1, embedding2):
        """Computes Manhattan (L1) distance between two embeddings."""
        return tf.reduce_sum(tf.abs(embedding1 - embedding2), axis=-1, keepdims=True)

    @staticmethod
    def absolute_difference(embedding1, embedding2):
        """Computes the absolute difference between two embeddings."""
        return tf.abs(embedding1 - embedding2)

    @staticmethod
    def ann(input_dim):
        """Creates an ANN-based similarity metric."""
        model = Sequential([
            Input(shape=(input_dim * 2,)),
            Dense(128, activation='relu'),
            Dense(64, activation='relu'),
            Dense(1, activation='sigmoid')  # Output similarity score (0 to 1)
        ])
        return model


In [ ]:


class LossFunction:
    """A unified class to handle different loss functions for a Siamese Network."""

    def __init__(self, loss_type="contrastive", margin=1.0, alpha=0.2):
        """
        Initialize the loss function.

        Parameters:
        - loss_type (str): The type of loss function ('contrastive', 'triplet', 'multilet').
        - margin (float): The margin value used in contrastive and triplet loss.
        - alpha (float): The scaling factor used in certain loss functions.
        """
        self.loss_type = loss_type.lower()
        self.margin = margin
        self.alpha = alpha

    def compute_loss(self, y_true, y_pred):
        """Computes the selected loss function dynamically."""
        if self.loss_type == "contrastive":
            return self._contrastive_loss(y_true, y_pred)
        elif self.loss_type == "triplet":
            return self._triplet_loss(y_true, y_pred)
        elif self.loss_type == "multilet":
            return self._multilet_loss(y_true, y_pred)
        else:
            raise ValueError(f"Unknown loss function: {self.loss_type}")

    def _contrastive_loss(self, y_true, y_pred):
        """Computes Contrastive Loss."""
        square_pred = tf.square(y_pred)
        margin_square = tf.square(tf.maximum(self.margin - y_pred, 0))
        return tf.reduce_mean(y_true * square_pred + (1 - y_true) * margin_square)

    def _triplet_loss(self, y_true, y_pred):
        """Computes Triplet Loss (y_pred contains [anchor, positive, negative])."""
        anchor, positive, negative = y_pred
        pos_dist = tf.reduce_sum(tf.square(anchor - positive), axis=-1)
        neg_dist = tf.reduce_sum(tf.square(anchor - negative), axis=-1)
        loss = tf.maximum(pos_dist - neg_dist + self.margin, 0.0)
        return tf.reduce_mean(loss)

    def _multilet_loss(self, y_true, y_pred):
        """Computes Multilet Loss using Huber loss."""
        return tf.reduce_mean(tf.losses.huber(y_true, y_pred))

    def __call__(self, y_true, y_pred):
        """Allows the class instance to be used as a function."""
        return self.compute_loss(y_true, y_pred)

    def get_config(self):
        return {
            "loss_type": self.loss_type,
            "margin": self.margin,
            "alpha": self.alpha
        }

In [ ]:
from tensorflow.keras.models import Model # type: ignore
from tensorflow.keras.layers import Input, Lambda # type: ignore
import tensorflow as tf


class SiameseNetwork:
    def __init__(self, feature_extractor, distance_metric="euclidean", loss_type="contrastive", input_shape=(128, 128, 3)):
        """
        Initializes the Siamese Network.

        Parameters:
        - feature_extractor: The feature extraction model
        - distance_metric (str): Distance metric ('euclidean', 'manhattan', 'absolute_difference')
        - loss_type (str): Loss function ('contrastive', 'triplet', 'multilet')
        - input_shape (tuple): Shape of the input images
        """
        self.feature_extractor = feature_extractor
        self.distance_metric = distance_metric.lower()
        self.loss_type = loss_type
        self.input_shape = input_shape
        self.model = self._build_model()

    def _build_model(self):
        """Builds the complete Siamese network architecture."""
        # Input layers for two images
        input_a = Input(shape=self.input_shape)
        input_b = Input(shape=self.input_shape)

        # Generate features using shared weights
        processed_a = self.feature_extractor(input_a)
        processed_b = self.feature_extractor(input_b)

        # Distance computation using DistanceMetrics class
        def get_distance(embeddings):
            if self.distance_metric == "euclidean":
                return DistanceMetrics.euclidean(embeddings[0], embeddings[1])
            elif self.distance_metric == "manhattan":
                return DistanceMetrics.manhattan(embeddings[0], embeddings[1])
            elif self.distance_metric == "absolute_difference":
                return DistanceMetrics.absolute_difference(embeddings[0], embeddings[1])
            else:
                raise ValueError(f"Unsupported distance metric: {self.distance_metric}")

        distance = Lambda(get_distance)([processed_a, processed_b])

        # Final model
        return Model(inputs=[input_a, input_b], outputs=distance)

    def compile_model(self, optimizer="adam"):
        """Compiles the model with the selected loss function."""
        loss = LossFunction(loss_type=self.loss_type)
        self.model.compile(optimizer=optimizer, loss=loss)

    def get_model(self):
        """ Returns the compiled siamese model """
        return self.model

In [ ]:
import os
import tensorflow as tf

class Trainer:
    """Handles training, evaluation, and saving/loading of a Siamese Network."""

    def __init__(self, siamese_network, save_dir="checkpoints", log_dir="logs"):
        """
        Initializes the trainer.

        Parameters:
        - siamese_network: Instance of the trained SiameseNetwork class.
        - save_dir (str): Directory to save trained model weights.
        - log_dir (str): Directory for TensorBoard logs.
        """
        self.model = siamese_network.get_model()
        self.save_dir = save_dir
        self.log_dir = log_dir

        # Ensure save directories exist
        os.makedirs(self.save_dir, exist_ok=True)
        os.makedirs(self.log_dir, exist_ok=True)

    def train(self, train_data, val_data=None, epochs=10, early_stop=True, checkpoint=True):
        callbacks = []

        if early_stop and val_data:
            callbacks.append(tf.keras.callbacks.EarlyStopping(
                monitor="val_loss", patience=5, restore_best_weights=True))

        if checkpoint:
            callbacks.append(tf.keras.callbacks.ModelCheckpoint(
                filepath=os.path.join(self.save_dir, "siamese_best.h5"),
                monitor="val_loss" if val_data else "loss",
                save_best_only=True,
                verbose=1
            ))

        callbacks.append(tf.keras.callbacks.TensorBoard(log_dir=self.log_dir))

        # Modified fit() - removed batch_size
        history = self.model.fit(
            train_data,
            validation_data=val_data,
            epochs=epochs,
            callbacks=callbacks
        )
        return history


    def evaluate(self, test_data):
        """
        Evaluates the trained model.

        Parameters:
        - test_data: Test dataset.

        Returns:
        - Loss and accuracy of the model on test set.
        """
        results = self.model.evaluate(test_data)
        print(f"Test Loss: {results}")
        return results

    def save_model(self, filename="siamese_final.h5"):
        """
        Saves the trained model.

        Parameters:
        - filename (str): Name of the file to save the model.
        """
        model_path = os.path.join(self.save_dir, filename)
        self.model.save(model_path)
        print(f"Model saved at: {model_path}")

    def load_model(self, filepath):
        """
        Loads a trained model.

        Parameters:
        - filepath (str): Path to the saved model.
        """
        self.model = tf.keras.models.load_model(filepath)
        print(f"Model loaded from: {filepath}")

In [ ]:


if __name__ == "__main__":
    # Dataset path
    train_path = r"/content/train_data/train"

    # Load dataset (returns unbatched)
    loader = DatasetLoader(train_path, target_size=(128, 128))
    dataset, class_mappings = loader.get_dataset()

    # Create pairs (now returns single pairs)
    train_pairs = create_balanced_pairs(
        dataset=dataset,
        class_mapping=class_mappings,
        samples_per_class=100
    )

    # Optional: Batch only at training time
    train_pairs = train_pairs.batch(32).prefetch(tf.data.AUTOTUNE)


    feature_extractor = FeatureExtractor(model_name='resnet', input_shape=(128, 128, 3))
    siamese_net = SiameseNetwork(
        feature_extractor=feature_extractor.get_model(),
        distance_metric='euclidean',
        loss_type='contrastive',
        input_shape=(128, 128, 3)
    )
    siamese_net.compile_model(optimizer='adam')

    trainer = Trainer(siamese_net)
    trainer.train(
        train_data=train_pairs,
        val_data=None,
        epochs=5,
        early_stop=False,
        checkpoint=False
    )
    trainer.save_model("siamese_resnet_bone_marrow.h5")

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
Epoch 1/5


/usr/local/lib/python3.11/dist-packages/keras/src/backend/tensorflow/trainer.py:82: UserWarning: The model does not have any trainable weights.
  warnings.warn("The model does not have any trainable weights.")


32/32 ━━━━━━━━━━━━━━━━━━━━ 17s 164ms/step - loss: 24.5380
Epoch 2/5


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/epoch_iterator.py:151: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


32/32 ━━━━━━━━━━━━━━━━━━━━ 12s 56ms/step - loss: 25.0432
Epoch 3/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 53ms/step - loss: 24.4336
Epoch 4/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 55ms/step - loss: 26.1265
Epoch 5/5
32/32 ━━━━━━━━━━━━━━━━━━━━ 3s 53ms/step - loss: 25.2703


Model saved at: checkpoints/siamese_resnet_bone_marrow.h5
